# Personalized Mobility Bundle Design and Selection

This notebook contains the implementation of a personalized mobility service bundling framework in two stages.

First, a mixed-integer optimization model is used to generate feasible bundles by selecting services, setting discount and usage levels, and balancing user utility, bundle cost, and persona-specific constraints. The model is solved for different mobility personas to generate candidate bundle options.

Second, the generated bundles are evaluated in a contextual bandit simulation. User contexts are simulated from persona-specific preference patterns, acceptance is modeled probabilistically, and a LinUCB policy is compared with a random policy to study adaptive bundle selection.

The notebook is organized as follows:
1. Define the service catalog, costs, personas, and bundle configuration levels.
2. Build the optimization model and solve it for a selected persona.
3. Generate multiple candidate bundles.
4. Simulate user contexts, acceptance, reward, and bandit learning.
5. Summarize and compare policy performance.

# Imports

In [10]:
import pyomo.environ as pyo
import math
import numpy as np
import pandas as pd
import mabwiser

## Service Catalog, Costs, and Persona Profiles

Each persona is characterized by:
- willingness to pay,
- maximum bundle size,
- discount limits,
- service weights,
- eligible services

In [11]:
# --- Services ---
L = [
    'PT','RH','BK','SC','CR',      # Transportation modes: transit, ridehailing, bike, scooter, car rental
    'CS','TR','FR','CRL','VP','AS','EB',  # carshare, tram, ferry, commuter rail, vanpool, airport shuttle, e-bike sub
    'PK','PKS','IF','RS','TL','INS'       # Supporting services: parking garage, street parking pass, infotainment, roadside, toll pass, insurance add-on
]

Mobility = {'PT','RH','BK','SC','CR','CS','TR','FR','CRL','VP','AS','EB'}
# (Supporting implicitly: PK, PKS, IF, RS, TL, INS)

# Monthly prices
Cost = {
    'PT':35,  'RH':28,  'BK':14,  'SC':12,  'CR':65,
    'CS':30,  'TR':25,  'FR':22,  'CRL':40, 'VP':18, 'AS':20, 'EB':16,
    'PK':35,  'PKS':15, 'IF':9,   'RS':12,  'TL':10, 'INS':12
}

# Base utility by asset type (1–5).
fA = {
    'PT':5, 'RH':4, 'BK':3, 'SC':3, 'CR':3,
    'CS':4, 'TR':4, 'FR':3, 'CRL':4, 'VP':3, 'AS':3, 'EB':3,
    'PK':2, 'PKS':2, 'IF':2, 'RS':3, 'TL':2, 'INS':2
}

#---------------------------Personas--------------------------------------------#
PERSONAS = {
    "Student": {
        "WTP": 45, "Lmax": 3, "q": 3, "Dbar_max": 0.20,
        "w": {'PT':5,'RH':3,'BK':4,'SC':4,'CR':0,'CS':3,'TR':4,'FR':3,'CRL':3,'VP':2,'AS':2,'EB':4,
              'PK':1,'PKS':1,'IF':2,'RS':2,'TL':1,'INS':1},
        "eligible": {'PT','BK','SC','EB'},
        "zones": {"Z1"}
    },
    "Professional": {
        "WTP": 100, "Lmax": 5, "q": 1, "Dbar_max": 0.01,
        "w": {'PT':3,'RH':5,'BK':1,'SC':2,'CR':3,'CS':3,'TR':4,'FR':3,'CRL':5,'VP':2,'AS':4,'EB':2,
              'PK':2,'PKS':1,'IF':2,'RS':3,'TL':2,'INS':3},
        "eligible": {'RH','CRL','AS'},
        "zones": {"Z1", "Z3"}
    },
    "Family": {
        "WTP": 160, "Lmax": 5, "q": 2, "Dbar_max": 0.15,
        "w": {'PT':4,'RH':3,'BK':3,'SC':2,'CR':3,'CS':3,'TR':3,'FR':3,'CRL':3,'VP':5,'AS':3,'EB':3,
              'PK':4,'PKS':3,'IF':1,'RS':3,'TL':3,'INS':3},
        "eligible": {'VP','PT','PK'},
        "zones": {"Z2"}
    },
}


persona_name = "Family"  # Options: "Student", "Professional", "Family"

persona_name = "Student"
pers = PERSONAS[persona_name]

WTP = float(pers["WTP"])
Lmax = int(pers["Lmax"])
q_max = int(pers["q"])
Dbar_max = float(pers["Dbar_max"])
w_persona = {l: float(pers["w"].get(l, 0.0)) for l in L}
eligible = set(pers.get("eligible", set(L)))

# Discounts, Number of Trips, Duration, and Providers

In [12]:
alpha = beta = gamma = 1.0 / 3.0

# --- discount/trips/duration level menus --
Dprime_levels = [0.00, 0.25]   # normalized for utility
Dpay_levels   = [0.00, 0.10]   # actual fractions
Tcost_levels  = [0.33, 0.67, 1.00]
Cprime_levels = [0.25, 0.50, 0.75, 1.00]
Tprime_levels = [0.33, 0.67, 1.00]
T_actual_days = {0.33: 7, 0.67: 14, 1.00: 30}  # 0.33→1 week, 0.67→2 weeks, 1.00→1 month

PROVIDERS = ["TransitCo", "RidehailCo", "MicromobCo", "AutoCo", "RailCo", "MarineCo", "PoolingCo", "SupportCo"]
prov_of = {
    'PT':"TransitCo",'TR':"TransitCo",
    'RH':"RidehailCo",
    'BK':"MicromobCo",'SC':"MicromobCo",'EB':"MicromobCo",
    'CR':"AutoCo",'CS':"AutoCo",
    'CRL':"RailCo",
    'FR':"MarineCo",
    'VP':"PoolingCo",'AS':"PoolingCo",
    'PK':"SupportCo",'PKS':"SupportCo",'IF':"SupportCo",'RS':"SupportCo",'TL':"SupportCo",'INS':"SupportCo"
}

a_map = {(q, l): int(prov_of[l] == q) for q in PROVIDERS for l in L}

# Main Optimization Block
The model:
- selects the services to include,
- assigns discount, trip, and duration levels,
- enforces persona-specific feasibility constraints,
- computes total utility and total cost,



In [13]:
# =========================
# Model builder
# =========================

def build_bundling_model():
    UTIL_SCALE = 25.0

    pi_u0 = 0.0
    pi_b0 = 0.0
    pi_q0 = {q: 0.0 for q in PROVIDERS}

    theta_u = 1.0
    theta_b = 1.0
    theta_q = {q: 1.0 for q in PROVIDERS}

    MU = 1.30
    kappa_mu = 0.5

    EPS = 1e-3
    pw_pts = [EPS, 0.05, 0.10, 0.20, 0.50, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 500.0]
    pw_max = pw_pts[-1]
    pw_pts_A = [EPS, 0.05, 0.10, 0.20, 0.50, 1.0]

    Dpay_val   = {i: v for i, v in enumerate(Dpay_levels)}
    Dprime_val = {i: v for i, v in enumerate(Dprime_levels)}
    Cprime_val = {i: v for i, v in enumerate(Cprime_levels)}
    Tprime_val = {i: v for i, v in enumerate(Tprime_levels)}
    Tcost_val  = {i: v for i, v in enumerate(Tcost_levels)}

    m = pyo.ConcreteModel()

    m.L = pyo.Set(initialize=L)
    m.Q = pyo.Set(initialize=PROVIDERS)

    m.MD = pyo.RangeSet(0, len(Dpay_levels) - 1)
    m.MC = pyo.RangeSet(0, len(Cprime_levels) - 1)
    m.MT = pyo.RangeSet(0, len(Tprime_levels) - 1)

    m.Cost = pyo.Param(m.L, initialize=Cost)
    m.fA = pyo.Param(m.L, initialize=fA)
    m.w = pyo.Param(m.L, initialize=w_persona)
    m.a = pyo.Param(m.Q, m.L, initialize=a_map, within=pyo.Binary)

    allowed = {l: int(l in eligible) for l in L}
    m.allowed = pyo.Param(m.L, initialize=allowed, within=pyo.Binary)

    m.x = pyo.Var(m.L, within=pyo.Binary)
    m.yD = pyo.Var(m.L, m.MD, within=pyo.Binary)
    m.yC = pyo.Var(m.L, m.MC, within=pyo.Binary)
    m.yT = pyo.Var(m.L, m.MT, within=pyo.Binary)

    m.p = pyo.Var(within=pyo.NonNegativeReals)
    m.r = pyo.Var(m.Q, within=pyo.NonNegativeReals)
    m.yq = pyo.Var(m.Q, within=pyo.Binary)

    m.Dprime = pyo.Expression(m.L, rule=lambda m_, l: sum(Dprime_val[i] * m_.yD[l, i] for i in m_.MD))
    m.Dpay = pyo.Expression(m.L, rule=lambda m_, l: sum(Dpay_val[i] * m_.yD[l, i] for i in m_.MD))
    m.Cprime = pyo.Expression(m.L, rule=lambda m_, l: sum(Cprime_val[i] * m_.yC[l, i] for i in m_.MC))
    m.Tprime = pyo.Expression(m.L, rule=lambda m_, l: sum(Tprime_val[i] * m_.yT[l, i] for i in m_.MT))

    m.u = pyo.Expression(
        m.L,
        rule=lambda m_, l: m_.fA[l] * (
            alpha * m_.Dprime[l] + beta * m_.Cprime[l] + gamma * m_.Tprime[l]
        )
    )

    m.U = pyo.Expression(
        rule=lambda m_: UTIL_SCALE * sum(m_.w[l] * m_.u[l] for l in m_.L)
    )

    m.Ctot = pyo.Expression(
        rule=lambda m_: sum(
            (1.0 - m_.Dpay[l]) * m_.Cost[l] *
            sum(Tcost_val[i] * m_.yT[l, i] for i in m_.MT)
            for l in m_.L
        )
    )

    m.Cq = pyo.Expression(
        m.Q,
        rule=lambda m_, q: sum(
            m_.a[q, l] * (1.0 - m_.Dpay[l]) * m_.Cost[l] *
            sum(Tcost_val[i] * m_.yT[l, i] for i in m_.MT)
            for l in m_.L
        )
    )

    m.con = pyo.ConstraintList()

    for l in L:
        m.con.add(sum(m.yD[l, i] for i in m.MD) == m.x[l])
        m.con.add(sum(m.yC[l, i] for i in m.MC) == m.x[l])
        m.con.add(sum(m.yT[l, i] for i in m.MT) == m.x[l])
        m.con.add(m.x[l] <= m.allowed[l])

    m.con.add(sum(m.x[l] for l in L) <= Lmax)
    m.con.add(sum(m.x[l] for l in L if l in Mobility) >= 1)

    m.con.add(sum(m.yD[l, i] for l in L for i in m.MD if Dpay_val[i] > 0.0) <= q_max)
    m.con.add(sum(m.Dpay[l] for l in L) <= Dbar_max * sum(m.x[l] for l in L))

    m.con.add(m.p >= m.Ctot)

    m.s_mu = pyo.Var(within=pyo.NonNegativeReals)
    m.con.add(m.p <= MU * m.Ctot + m.s_mu)
    m.con.add(m.p <= WTP)

    for q in PROVIDERS:
        Lq = [l for l in L if prov_of[l] == q]
        if not Lq:
            m.yq[q].fix(0)
        else:
            for l in Lq:
                m.con.add(m.yq[q] >= m.x[l])
            m.con.add(m.yq[q] <= sum(m.x[l] for l in Lq))

    m.con.add(sum(m.r[q] for q in PROVIDERS) <= m.p)

    m.Unet = pyo.Expression(rule=lambda m_: m_.U - m_.p)

    m.Su = pyo.Var(bounds=(0.0, pw_max - EPS))
    m.Sb = pyo.Var(bounds=(0.0, pw_max - EPS))
    m.Sq = pyo.Var(m.Q, bounds=(0.0, pw_max - EPS))
    m.Sq_eff = pyo.Var(m.Q, bounds=(EPS, pw_max))

    m.con.add(m.Su == m.Unet - pi_u0)
    m.con.add(m.Sb == (m.p - sum(m.r[q] for q in PROVIDERS)) - pi_b0)

    for q in PROVIDERS:
        m.con.add(m.Sq[q] == (m.r[q] - m.Cq[q]) - pi_q0[q])
        m.con.add(m.Sq[q] <= (pw_max - EPS) * m.yq[q])

        m.con.add(m.Sq_eff[q] <= (m.Sq[q] + EPS) + pw_max * (1 - m.yq[q]))
        m.con.add(m.Sq_eff[q] >= (m.Sq[q] + EPS) - pw_max * (1 - m.yq[q]))
        m.con.add(m.Sq_eff[q] <= 1.0 + pw_max * m.yq[q])
        m.con.add(m.Sq_eff[q] >= 1.0 - pw_max * m.yq[q])

    m.Su_eff = pyo.Var(bounds=(EPS, pw_max))
    m.Sb_eff = pyo.Var(bounds=(EPS, pw_max))

    m.con.add(m.Su_eff == m.Su + EPS)
    m.con.add(m.Sb_eff == m.Sb + EPS)

    m.z = pyo.Var(bounds=(0.0, 1.0))
    m.con.add(m.z == m.p / WTP)

    m.A = pyo.Var(bounds=(0.0, 1.0))

    z_pts = [0.0, 0.10, 0.20, 0.35, 0.50, 0.65, 0.80, 0.90, 1.0]

    m.pw_Amap = pyo.Piecewise(
        m.A, m.z,
        pw_pts=z_pts,
        f_rule=lambda m_, x: 1.0 - x * x,
        pw_constr_type="EQ",
        pw_repn="SOS2"
    )

    m.A_eff = pyo.Var(bounds=(EPS, 1.0 + EPS))
    m.con.add(m.A_eff == m.A + EPS)

    m.log_u = pyo.Var(bounds=(math.log(pw_pts[0]), math.log(pw_pts[-1])))
    m.log_b = pyo.Var(bounds=(math.log(pw_pts[0]), math.log(pw_pts[-1])))
    m.log_q = pyo.Var(m.Q, bounds=(math.log(pw_pts[0]), math.log(pw_pts[-1])))
    m.log_A = pyo.Var(bounds=(math.log(pw_pts_A[0]), math.log(pw_pts_A[-1])))

    m.pw_u = pyo.Piecewise(
        m.log_u, m.Su_eff,
        pw_pts=pw_pts,
        f_rule=lambda m_, x: math.log(x),
        pw_constr_type="EQ",
        pw_repn="SOS2"
    )

    m.pw_b = pyo.Piecewise(
        m.log_b, m.Sb_eff,
        pw_pts=pw_pts,
        f_rule=lambda m_, x: math.log(x),
        pw_constr_type="EQ",
        pw_repn="SOS2"
    )

    m.pw_q = pyo.Piecewise(
        m.Q, m.log_q, m.Sq_eff,
        pw_pts=pw_pts,
        f_rule=lambda m_, q_, x: math.log(x),
        pw_constr_type="EQ",
        pw_repn="SOS2"
    )

    m.pw_A = pyo.Piecewise(
        m.log_A, m.A_eff,
        pw_pts=pw_pts_A,
        f_rule=lambda m_, x: math.log(x),
        pw_constr_type="EQ",
        pw_repn="SOS2"
    )

    m.obj = pyo.Objective(
        expr=
            theta_u * m.log_u
            + theta_b * (m.log_b + m.log_A)
            + sum(theta_q[q] * m.log_q[q] for q in PROVIDERS)
            - kappa_mu * m.s_mu,
        sense=pyo.maximize
    )

    return m


# =========================
# Solver wrapper
# =========================

def solve_model(model, solver_name="gurobi", tee=False):
    solver = pyo.SolverFactory(solver_name)

    if not solver.available():
        raise RuntimeError(f"{solver_name} is not available.")

    if solver_name == "gurobi":
        solver.options["MIPGap"] = 0
        solver.options["Threads"] = 1
        solver.options["Seed"] = 0

    result = solver.solve(model, tee=tee)

    tc = result.solver.termination_condition
    if tc not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        raise RuntimeError(f"Solve failed: {tc}")

    return result


# =========================
# Reporting
# =========================

def print_solution(model):
    picked = [l for l in L if pyo.value(model.x[l]) > 0.5]

    print("\nPersona:", persona_name)
    print("Chosen services:", picked)
    print("p:", round(pyo.value(model.p), 2))
    print("Ctot:", round(pyo.value(model.Ctot), 2))
    print("s_mu:", round(pyo.value(model.s_mu), 4))
    print("U:", round(pyo.value(model.U), 4))
    print("A:", round(pyo.value(model.A), 4))
    print("Su:", round(pyo.value(model.Su), 4))
    print("Sb:", round(pyo.value(model.Sb), 4))

    for q in PROVIDERS:
        if pyo.value(model.yq[q]) > 0.5:
            print(
                q,
                "r:", round(pyo.value(model.r[q]), 2),
                "Cq:", round(pyo.value(model.Cq[q]), 2),
                "Sq:", round(pyo.value(model.Sq[q]), 4)
            )


# =========================
# Run
# =========================

m = build_bundling_model()
res = solve_model(m, solver_name="gurobi", tee=False)
print_solution(m)

**WARNING: Piecewise 'pw_A[None]' feasible region does not include the upper bound of domain variable: A_eff.ub = 1.001 > 1.0. Refer to the Piecewise help documentation for information on how to disable this warning.

Persona: Student
Chosen services: ['PT', 'BK']
p: 31.71
Ctot: 24.4
s_mu: 0.0
U: 529.1667
A: 0.4981
Su: 497.4532
Sb: 1.999
TransitCo r: 12.39 Cq: 10.4 Sq: 1.999
MicromobCo r: 17.32 Cq: 14.0 Sq: 3.3205


## Optimization Model for Repeated Bundle Generation

We rebuild the optimization model in a form that can be solved repeatedly.

It is used to generate multiple candidate bundles by solving the model, storing each solution, and then adding an exclusion constraint so that the next solution differs from the previous one.

In [14]:
# =========================
# Candidate generation
# =========================

def extract_solution(model, candidate_id=None):
    picked = [l for l in L if pyo.value(model.x[l]) > 0.5]

    active_providers = []
    provider_details = {}

    for q in PROVIDERS:
        if pyo.value(model.yq[q]) > 0.5:
            active_providers.append(q)
            provider_details[q] = {
                "r": pyo.value(model.r[q]),
                "Cq": pyo.value(model.Cq[q]),
                "Sq": pyo.value(model.Sq[q]),
            }

    solution = {
        "candidate_id": candidate_id,
        "persona": persona_name,
        "services": picked,
        "p": pyo.value(model.p),
        "Ctot": pyo.value(model.Ctot),
        "s_mu": pyo.value(model.s_mu),
        "U": pyo.value(model.U),
        "A": pyo.value(model.A),
        "Su": pyo.value(model.Su),
        "Sb": pyo.value(model.Sb),
        "active_providers": active_providers,
        "provider_details": provider_details,
        "objective": pyo.value(model.obj),
    }

    return solution


def add_bundle_exclusion_cut(model, picked):
    selected = set(picked)

    model.con.add(
        sum(model.x[l] for l in selected)
        - sum(model.x[l] for l in L if l not in selected)
        <= len(selected) - 1
    )


def generate_candidate_bundles(K=5, solver_name="gurobi", tee=False):
    model = build_bundling_model()
    candidates = []

    for k in range(1, K + 1):
        try:
            solve_model(model, solver_name=solver_name, tee=tee)
        except RuntimeError as e:
            print(f"Stopped at candidate {k}: {e}")
            break

        sol = extract_solution(model, candidate_id=k)
        candidates.append(sol)

        picked = sol["services"]

        if len(picked) == 0:
            break

        add_bundle_exclusion_cut(model, picked)

    return candidates, model


def print_candidates(candidates):
    for sol in candidates:
        print("\nCandidate:", sol["candidate_id"])
        print("Persona:", sol["persona"])
        print("Services:", sol["services"])
        print("p:", round(sol["p"], 2))
        print("Ctot:", round(sol["Ctot"], 2))
        print("s_mu:", round(sol["s_mu"], 4))
        print("U:", round(sol["U"], 4))
        print("A:", round(sol["A"], 4))
        print("Su:", round(sol["Su"], 4))
        print("Sb:", round(sol["Sb"], 4))
        print("Objective:", round(sol["objective"], 4))

        for q, vals in sol["provider_details"].items():
            print(
                q,
                "r:", round(vals["r"], 2),
                "Cq:", round(vals["Cq"], 2),
                "Sq:", round(vals["Sq"], 4)
            )


# =========================
# Run candidate generation
# =========================


candidates, candidate_model = generate_candidate_bundles(K=5, solver_name="gurobi", tee=False)
print_candidates(candidates)

**WARNING: Piecewise 'pw_A[None]' feasible region does not include the upper bound of domain variable: A_eff.ub = 1.001 > 1.0. Refer to the Piecewise help documentation for information on how to disable this warning.

Candidate: 1
Persona: Student
Services: ['PT', 'BK']
p: 31.71
Ctot: 24.4
s_mu: 0.0
U: 529.1667
A: 0.4981
Su: 497.4532
Sb: 1.999
Objective: 7.9941
TransitCo r: 12.39 Cq: 10.4 Sq: 1.999
MicromobCo r: 17.32 Cq: 14.0 Sq: 3.3205

Candidate: 2
Persona: Student
Services: ['PT', 'BK', 'EB']
p: 31.46
Ctot: 24.2
s_mu: 0.0
U: 529.1667
A: 0.5062
Su: 497.7054
Sb: 1.999
Objective: 7.9898
TransitCo r: 12.39 Cq: 10.4 Sq: 1.999
MicromobCo r: 17.07 Cq: 13.81 Sq: 3.2623

Candidate: 3
Persona: Student
Services: ['PT', 'SC', 'EB']
p: 31.64
Ctot: 24.34
s_mu: 0.0
U: 527.0833
A: 0.5003
Su: 495.4387
Sb: 1.999
Objective: 7.9876
TransitCo r: 13.55 Cq: 11.55 Sq: 1.999
MicromobCo r: 16.1 Cq: 12.79 Sq: 3.3046

Candidate: 4
Persona: Student
Services: ['PT', 'BK', 'SC']
p: 31.84
Ctot: 24.49
s_mu: 0.0
U:

## Contextual Bandit Simulation

We now evaluate the generated bundles using a contextual bandit model.

The code simulates user contexts, models acceptance probability, defines the reward structure, and compares a random policy with a LinUCB policy. The goal is to see whether adaptive bundle selection improves performance under heterogeneous user preferences.

In [15]:
# =========================
# Candidate checks
# =========================

if "candidates" not in globals():
    raise RuntimeError("Run generate_candidate_bundles(...) first to create candidates.")

if len(candidates) == 0:
    raise RuntimeError("Candidate list is empty.")

required_keys = {"services", "p", "Sb", "provider_details"}
for i, cand in enumerate(candidates):
    missing = required_keys - set(cand.keys())
    if missing:
        raise RuntimeError(f"Candidate {i} is missing keys: {missing}")


# =========================
# Service groups
# =========================

SERVICE_GROUPS = {
    "transit": {"PT", "TR", "CRL", "FR"},
    "micromobility": {"BK", "SC", "EB"},
    "auto": {"RH", "CR", "CS", "VP", "AS"},
    "support": {"PK", "PKS", "IF", "RS", "TL", "INS"},
}

CONTEXT_FEATURES = [
    "intercept",
    "wtp_norm",
    "travel_frequency",
    "pref_transit",
    "pref_micromobility",
    "pref_auto",
    "pref_support",
]


# =========================
# Context simulation
# =========================

def sample_user_context(persona_name, base_wtp):
    if persona_name == "Student":
        pref_probs = {
            "pref_transit": 0.80,
            "pref_micromobility": 0.85,
            "pref_auto": 0.20,
            "pref_support": 0.25,
        }
        freq_prob = 0.80

    elif persona_name == "Professional":
        pref_probs = {
            "pref_transit": 0.55,
            "pref_micromobility": 0.25,
            "pref_auto": 0.75,
            "pref_support": 0.45,
        }
        freq_prob = 0.75

    elif persona_name == "Family":
        pref_probs = {
            "pref_transit": 0.55,
            "pref_micromobility": 0.30,
            "pref_auto": 0.65,
            "pref_support": 0.80,
        }
        freq_prob = 0.65

    else:
        pref_probs = {
            "pref_transit": 0.50,
            "pref_micromobility": 0.50,
            "pref_auto": 0.50,
            "pref_support": 0.50,
        }
        freq_prob = 0.50

    wtp_user = np.random.normal(base_wtp, 0.15 * base_wtp)
    wtp_user = max(0.60 * base_wtp, min(1.40 * base_wtp, wtp_user))

    context = {
        "wtp": wtp_user,
        "travel_frequency": int(np.random.random() < freq_prob),
    }

    for key, prob in pref_probs.items():
        context[key] = int(np.random.random() < prob)

    return context


def context_to_vector(context, base_wtp):
    return np.array([
        1.0,
        context["wtp"] / base_wtp,
        context["travel_frequency"],
        context["pref_transit"],
        context["pref_micromobility"],
        context["pref_auto"],
        context["pref_support"],
    ], dtype=float)


# =========================
# Acceptance simulation
# =========================

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def bundle_group_flags(candidate):
    services = set(candidate["services"])

    flags = {}
    for group, service_set in SERVICE_GROUPS.items():
        flags[group] = int(len(services.intersection(service_set)) > 0)

    return flags


def preference_match_score(context, candidate):
    flags = bundle_group_flags(candidate)

    prefs = {
        "transit": context["pref_transit"],
        "micromobility": context["pref_micromobility"],
        "auto": context["pref_auto"],
        "support": context["pref_support"],
    }

    total_pref = sum(prefs.values())
    if total_pref == 0:
        return 0.0

    matched = sum(prefs[g] * flags[g] for g in prefs)
    return matched / total_pref


def acceptance_probability(context, candidate):
    price_ratio = candidate["p"] / context["wtp"]
    match = preference_match_score(context, candidate)
    freq = context["travel_frequency"]

    eta_0 = -0.50
    eta_match = 1.60
    eta_freq = 0.60
    eta_price = 2.20

    score = (
        eta_0
        + eta_match * match
        + eta_freq * freq
        - eta_price * price_ratio
    )

    prob = sigmoid(score)
    return float(np.clip(prob, 0.01, 0.99))


# =========================
# Reward simulation
# =========================

def provider_surplus_sum(candidate):
    details = candidate.get("provider_details", {})
    return sum(vals["Sq"] for vals in details.values())


def candidate_reward_if_accepted(candidate):
    rho_q = 0.20
    return candidate["Sb"] + rho_q * provider_surplus_sum(candidate)


def simulate_reward(context, candidate):
    prob = acceptance_probability(context, candidate)
    accepted = int(np.random.random() < prob)
    reward = accepted * candidate_reward_if_accepted(candidate)

    return reward, accepted, prob


def expected_reward(context, candidate):
    prob = acceptance_probability(context, candidate)
    return prob * candidate_reward_if_accepted(candidate)


# =========================
# LinUCB policy
# =========================

class LinUCB:
    def __init__(self, n_actions, n_features, alpha=1.0):
        self.n_actions = n_actions
        self.n_features = n_features
        self.alpha = alpha
        self.A = [np.eye(n_features) for _ in range(n_actions)]
        self.b = [np.zeros(n_features) for _ in range(n_actions)]

    def choose(self, x):
        scores = []

        for a in range(self.n_actions):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]
            mean = theta @ x
            bonus = self.alpha * np.sqrt(x @ A_inv @ x)
            scores.append(mean + bonus)

        return int(np.argmax(scores))

    def update(self, action, x, reward):
        self.A[action] += np.outer(x, x)
        self.b[action] += reward * x


# =========================
# Simulation runners
# =========================

def run_random_policy(candidates, T=1000):
    records = []

    for t in range(T):
        context = sample_user_context(persona_name, WTP)
        action = np.random.randint(len(candidates))
        candidate = candidates[action]

        reward, accepted, prob = simulate_reward(context, candidate)

        oracle_expected = max(expected_reward(context, c) for c in candidates)
        chosen_expected = expected_reward(context, candidate)

        records.append({
            "t": t,
            "policy": "Random",
            "action": action,
            "candidate_id": candidate.get("candidate_id", action),
            "reward": reward,
            "accepted": accepted,
            "accept_prob": prob,
            "chosen_expected": chosen_expected,
            "oracle_expected": oracle_expected,
            "regret": oracle_expected - chosen_expected,
            "price": candidate["p"],
            "Sb": candidate["Sb"],
            "provider_surplus_sum": provider_surplus_sum(candidate),
            "reward_if_accepted": candidate_reward_if_accepted(candidate),
            "services": tuple(candidate["services"]),
        })

    return pd.DataFrame(records)


def run_linucb_policy(candidates, T=1000, alpha=1.0):
    n_actions = len(candidates)
    n_features = len(CONTEXT_FEATURES)

    policy = LinUCB(n_actions=n_actions, n_features=n_features, alpha=alpha)
    records = []

    for t in range(T):
        context = sample_user_context(persona_name, WTP)
        x = context_to_vector(context, WTP)

        action = policy.choose(x)
        candidate = candidates[action]

        reward, accepted, prob = simulate_reward(context, candidate)
        policy.update(action, x, reward)

        oracle_expected = max(expected_reward(context, c) for c in candidates)
        chosen_expected = expected_reward(context, candidate)

        records.append({
            "t": t,
            "policy": "LinUCB",
            "action": action,
            "candidate_id": candidate.get("candidate_id", action),
            "reward": reward,
            "accepted": accepted,
            "accept_prob": prob,
            "chosen_expected": chosen_expected,
            "oracle_expected": oracle_expected,
            "regret": oracle_expected - chosen_expected,
            "price": candidate["p"],
            "Sb": candidate["Sb"],
            "provider_surplus_sum": provider_surplus_sum(candidate),
            "reward_if_accepted": candidate_reward_if_accepted(candidate),
            "services": tuple(candidate["services"]),
        })

    return pd.DataFrame(records)


# =========================
# Reporting
# =========================

def summarize_bandit_results(df):
    summary = (
        df.groupby("policy")
        .agg(
            avg_reward=("reward", "mean"),
            acceptance_rate=("accepted", "mean"),
            avg_accept_prob=("accept_prob", "mean"),
            avg_price=("price", "mean"),
            avg_bundler_surplus=("Sb", "mean"),
            avg_provider_surplus_sum=("provider_surplus_sum", "mean"),
            avg_reward_if_accepted=("reward_if_accepted", "mean"),
            avg_expected_reward=("chosen_expected", "mean"),
            cumulative_regret=("regret", "sum"),
        )
        .reset_index()
    )

    return summary


def action_frequency(df):
    freq = (
        df.groupby(["policy", "candidate_id", "services"])
        .size()
        .reset_index(name="count")
    )
    freq["share"] = freq.groupby("policy")["count"].transform(lambda x: x / x.sum())
    return freq.sort_values(["policy", "count"], ascending=[True, False])


# =========================
# Run experiment
# =========================

T = 1000

df_random = run_random_policy(candidates, T=T)
df_linucb = run_linucb_policy(candidates, T=T, alpha=1.0)

df_results = pd.concat([df_random, df_linucb], ignore_index=True)

summary = summarize_bandit_results(df_results)
freq = action_frequency(df_results)

print("\nBandit Summary")
print(summary)

print("\nAction Selection Frequency")
print(freq)


Bandit Summary
   policy  avg_reward  acceptance_rate  avg_accept_prob  avg_price  \
0  LinUCB    1.372586            0.447         0.426997  31.862029   
1  Random    1.263541            0.412         0.428603  31.777976   

   avg_bundler_surplus  avg_provider_surplus_sum  avg_reward_if_accepted  \
0                1.999                  5.353776                3.069755   
1                1.999                  5.334379                3.065876   

   avg_expected_reward  cumulative_regret  
0             1.310950           6.071661  
1             1.313982           4.947690  

Action Selection Frequency
   policy  candidate_id      services  count  share
3  LinUCB             4  (PT, BK, SC)    523  0.523
0  LinUCB             1      (PT, BK)    233  0.233
4  LinUCB             5      (PT, EB)    173  0.173
2  LinUCB             3  (PT, SC, EB)     50  0.050
1  LinUCB             2  (PT, BK, EB)     21  0.021
7  Random             3  (PT, SC, EB)    210  0.210
6  Random           